# Lesson 6 — Branch and Bound, Conceptually

## Learning objectives

1. Trace by hand a B&B tree for a small MILP.
2. Distinguish *bounding*, *branching*, and *node selection*.
3. Read a `discopt` B&B log and identify when nodes are pruned by bound, by integrality, or by infeasibility.
4. Reason about why some MILPs blow up the tree and others close in 1 node.

## 1. The algorithm in pseudocode

```
LB = -inf, UB = +inf, incumbent = None
push root LP relaxation onto stack

while stack non-empty:
    pop node
    solve LP relaxation -> z_LP
    if infeasible:                  prune (infeasibility)
    elif z_LP >= UB:                prune (bound)
    elif x_LP is integer:           if z_LP < UB: UB = z_LP; incumbent = x_LP
    else:                           branch on a fractional variable, push children
```

This is {cite:p}`Land1960` and {cite:p}`Dakin1965`. Modern solvers add presolve, cuts, heuristics, and clever branching/node selection on top — but this is the core.

## 2. Branching choices

When $x_j^\star$ in the LP solution is fractional ($x_j = 1.7$, say), branch:

- **Down child:** add $x_j \le 1$.
- **Up child:** add $x_j \ge 2$.

Which fractional variable to pick is the *branching rule*. Lesson 14 covers strong branching {cite:p}`Achterberg2005`. For now: most fractional, or pseudo-cost based, are common heuristics.

## 3. Node selection

Order in which to process the open nodes:

- **Best-first** (lowest LP bound first) — minimizes nodes processed.
- **Depth-first** — minimizes memory; finds incumbents fast.
- **Best-estimate / hybrid** — balances both.

`discopt` defaults to best-estimate {cite:p}`Achterberg2007`.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
import numpy as np, discopt as do

# Small knapsack we'll trace by hand
v = np.array([10, 14, 12, 11])
w = np.array([5, 7, 6, 5])
W = 13

m = do.Model("k4")
x = m.binary("x", shape=(4,))
m.subject_to(w @ x <= W)
m.maximize(v @ x)
r = m.solve()

print("optimal:", r.objective, "  x*:", r.value(x))
print("nodes processed:", r.node_count, "  wall time:", round(r.wall_time, 4))
print("dual bound:", r.bound, "  gap:", r.gap)


## 4. Pruning logic

A node is pruned by:

- **Infeasibility:** LP relaxation has no solution.
- **Bound:** LP value is worse than the incumbent ($z_{LP} \ge \text{UB}$ for min).
- **Integrality:** LP solution is integer; update incumbent and prune.

Stronger LP relaxations → more pruning by bound → smaller tree. Hence the obsession with cuts (Lesson 13) and tight formulations (Lesson 5).

## 5. Why some MILPs explode

Two pathologies:

- **Symmetry** — equivalent feasible solutions reached by swapping variable indices. The tree must still explore branches that lead to identical solutions {cite:p}`Margot2010`.
- **Weak bounds** — LP relaxation far above the IP optimum, so few prunings happen.

Both are addressable: symmetry breaking via lex constraints; bounds via tighter formulation, presolve, and cuts.

## References
{cite:p}`Land1960,Dakin1965` (originals), {cite:p}`Wolsey1998` (textbook), {cite:p}`Achterberg2007` (modern engineering).